In [8]:
%pip install scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['long_call'] = (df['duration'] > 300).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA = add_features(TEST_DATA)

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'long_call']

def preprocess(df, encoder, scaler, poly, fit=False):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(),
        index=df.index
    )
    num_scaled = scaler.transform(df[num_cols])
    
    # Create interaction terms between numerical features only
    num_poly = poly.transform(num_scaled)
    num_poly_df = pd.DataFrame(
        num_poly,
        columns=poly.get_feature_names_out(num_cols),
        index=df.index
    )
    return pd.concat([cat_enc, num_poly_df], axis=1)

ENCODER = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(TRAIN_DATA[cat_cols])
SCALER = StandardScaler().fit(TRAIN_DATA[num_cols])
POLY = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False).fit(
    SCALER.transform(TRAIN_DATA[num_cols])
)

X_train = preprocess(TRAIN_DATA, ENCODER, SCALER, POLY)
X_test = preprocess(TEST_DATA, ENCODER, SCALER, POLY)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 98)
X_test shape: (19893, 98)


In [11]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# RepeatedStratifiedKFold = k-Fold repeated multiple times for more stable estimate
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.1,
    random_state=42
)

cv_scores = cross_val_score(model, X_train, y_train, cv=rskf, scoring='balanced_accuracy', n_jobs=-1)
print(f'CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold scores: {cv_scores}')

CV Balanced Accuracy: 0.8169 ± 0.0058
Per-fold scores: [0.81401292 0.82209945 0.81764123 0.81920614 0.80912204 0.82247304
 0.80761003 0.82509342 0.81139281 0.8224904  0.8112265  0.82585229
 0.81510373 0.81954296 0.81094784]


In [12]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
I_train, I_val = next(splitter.split(X_train, y_train))
X_tr, X_val = X_train.iloc[I_train], X_train.iloc[I_val]
y_tr, y_val = y_train[I_train], y_train[I_val]

model.fit(X_tr, y_tr)
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold, best_ba = 0.5, 0.0
for thresh in np.arange(0.1, 0.9, 0.01):
    preds = (val_probs >= thresh).astype(int)
    ba = balanced_accuracy_score(y_val, preds)
    if ba > best_ba:
        best_ba = ba
        best_threshold = thresh

print(f'Best threshold: {best_threshold:.2f}')
print(f'Best Val Balanced Accuracy: {best_ba:.4f}')

Best threshold: 0.42
Best Val Balanced Accuracy: 0.8306


In [13]:
final_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.1,
    random_state=42
)

final_model.fit(X_train, y_train)

test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print(f'Prediction distribution ~0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')

Prediction distribution ~0: 13779, 1: 6114


save to csv


In [14]:
submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'subscription': test_preds
})

submission.to_csv('submission.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             0
